# EC2 Tension Shift Rule Demonstration

This notebook demonstrates the EC2 §9.2.1.3 tension shift rule using interactive visualizations.

## Background

The tension shift rule accounts for the additional tensile force in the longitudinal reinforcement 
caused by the diagonal compression strut in the shear truss model. This shifts the tensile force 
diagram horizontally towards regions of higher moment, requiring either:
- Increased reinforcement at beam ends
- Proper anchorage of the shifted force envelope

### EC2 §9.2.1.3 Summary

The shift distance $a_l$ is:
- **Without shear reinforcement**: $a_l = d$ (effective depth)
- **With shear reinforcement**: $a_l = 0.5 \cdot z \cdot \cot\theta$ (lever arm × strut angle)

Where:
- $d$ = effective depth to tension reinforcement
- $z$ = internal lever arm (~0.9d for beams)
- $\theta$ = strut angle from shear design (typically 21.8° to 45°)

## Contents

1. [Setup and Imports](#setup)
2. [Create RC Section and Interaction Diagram](#section)
3. [Simply Supported Beam Example](#simply-supported)
4. [Apply Tension Shift using the API](#tension-shift)
5. [Comparison Visualization](#comparison)
6. [Fixed-Pinned Beam with Shear Reinforcement](#fixed-pinned)

## 1. Setup and Imports <a id='setup'></a>

In [1]:
# Setup: Add project root to Python path
import sys
from pathlib import Path

# Notebook is in examples/ subdirectory, go up one level to project root
project_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"[OK] Project root: {project_root}")

[OK] Project root: c:\Users\user\Repo\Scripts\section_design_checks


In [2]:
# Standard imports
import numpy as np

# Beam analysis module
from examples.beam_analysis import (
    SimplySupportedBeam,
    FixedPinnedBeam,
    plot_beam_diagrams,
    plot_tension_shift_comparison,
    apply_tension_shift_to_beam,  # Uses actual MNInteractionDiagram API
)

# Materials library imports
from materials.reinforced_concrete.constitutive import ConcreteModelType, SteelModelType
from materials.reinforced_concrete.materials import ConcreteMaterial, Rebar
from materials.reinforced_concrete.materials.rebar import ShearRebar
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_linear_rebar_layer,
)
from materials.reinforced_concrete.analysis import create_interaction_diagram

print("[OK] All imports successful")
print("[OK] Interactive Plotly visualizations available")

[OK] All imports successful
[OK] Interactive Plotly visualizations available


## 2. Create RC Section and Interaction Diagram <a id='section'></a>

We create a 300×500 mm beam section that will be used to calculate tension shift.

The `MNInteractionDiagram.apply_tension_shift()` method:
1. Solves for the internal lever arm z from strain compatibility
2. Calculates cot(θ) from shear design equations
3. Computes shift distance a_l
4. Returns the shifted design moment M_design

In [3]:
# Create materials
concrete = ConcreteMaterial(grade="C30/37", gamma_c=1.5, alpha_cc=0.85)
rebar = Rebar(grade="B500B", diameter=20)

print("Materials:")
print(f"  Concrete: {concrete.grade}, f_cd = {concrete.f_cd:.1f} MPa")
print(f"  Steel: {rebar.grade}, f_yd = {rebar.f_yd:.1f} MPa")

Materials:
  Concrete: C30/37, f_cd = 17.0 MPa
  Steel: B500B, f_yd = 434.8 MPa


In [4]:
# Create 300×500 mm beam section
section_width = 300  # mm
section_height = 500  # mm
cover = 50  # mm

section = create_rectangular_section(
    width=section_width,
    height=section_height,
    section_name="Demo Beam 300×500",
)

# Calculate rebar positions
r = rebar.diameter / 2
x_left = cover + r
x_right = section_width - cover - r
y_bottom = cover + r
y_top = section_height - cover - r

# Add bottom reinforcement (tension for sagging)
bottom_layer = create_linear_rebar_layer(
    rebar=rebar,
    n_bars=3,
    start_point=(x_left, y_bottom),
    end_point=(x_right, y_bottom),
    layer_name="bottom",
)
section.add_rebar_group(bottom_layer)

# Add top reinforcement (tension for hogging)
top_layer = create_linear_rebar_layer(
    rebar=rebar,
    n_bars=2,
    start_point=(x_left, y_top),
    end_point=(x_right, y_top),
    layer_name="top",
)
section.add_rebar_group(top_layer)
section.plot()

print("RC Section Created:")
print(f"  Dimensions: {section_width}×{section_height} mm")
print(f"  Cover: {cover} mm")
print(f"  Bottom reinforcement: 3×ϕ{rebar.diameter:.0f} at y = {y_bottom:.0f} mm")
print(f"  Top reinforcement: 2×ϕ{rebar.diameter:.0f} at y = {y_top:.0f} mm")
print(f"  Effective depth d ≈ {section_height - y_bottom:.0f} mm")

RC Section Created:
  Dimensions: 300×500 mm
  Cover: 50 mm
  Bottom reinforcement: 3×ϕ20 at y = 60 mm
  Top reinforcement: 2×ϕ20 at y = 440 mm
  Effective depth d ≈ 440 mm


In [5]:
# Create M-N interaction diagram
diagram = create_interaction_diagram(
    section=section,
    concrete=concrete,
    concrete_model_type=ConcreteModelType.PARABOLA_RECTANGLE,
    steel_model_type=SteelModelType.INCLINED,
    n_fibres_width=30,
    n_fibres_height=50,
)

print(diagram)
print()
print(f"Fibre mesh: {diagram.mesh.total_fibres} fibres")
print(f"  Concrete: {len(diagram.mesh.concrete_fibres)}")
print(f"  Steel: {len(diagram.mesh.steel_fibres)}")

MNInteractionDiagram(section=Demo Beam 300×500, concrete=C30/37, fibres=1505, tension_stiffening=False, confined=False)

Fibre mesh: 1505 fibres
  Concrete: 1500
  Steel: 5


In [6]:
# Display the M-N interaction diagram
fig_mn = diagram.plot_mn(
    title="M-N Interaction Diagram<br>300×500 mm Beam, C30/37",
    n_points=100,
    show=True,
)

## 3. Simply Supported Beam Example <a id='simply-supported'></a>

A simply supported beam with uniformly distributed load (UDL).

**Properties:**
- Span: L = 8.0 m
- UDL: w = 25.0 kN/m (downward)
- Maximum moment at midspan: $M_{max} = \frac{wL^2}{8}$

In [7]:
# Create simply supported beam
ss_beam = SimplySupportedBeam(
    length_m=8.0,
    udl_kN_m=25.0,
    n_nodes=101,
)

# Display results
print("Simply Supported Beam Analysis")
print("=" * 40)
print(f"Span: {ss_beam.length_m} m")
print(f"UDL: {ss_beam.udl_kN_m} kN/m")
print(f"Number of nodes: {ss_beam.n_nodes}")
print()

# Key results
M_max_pos, x_max = ss_beam.get_max_positive_moment()
w, L = ss_beam.udl_kN_m, ss_beam.length_m

print(f"Maximum positive moment: {M_max_pos:.2f} kN·m at x = {x_max:.2f} m")
print(f"Theoretical M_max = wL²/8 = {w * L**2 / 8:.2f} kN·m")
print(f"Maximum shear at supports: ±{w * L / 2:.2f} kN")

Simply Supported Beam Analysis
Span: 8.0 m
UDL: 25.0 kN/m
Number of nodes: 101

Maximum positive moment: 200.00 kN·m at x = 4.00 m
Theoretical M_max = wL²/8 = 200.00 kN·m
Maximum shear at supports: ±100.00 kN


In [8]:
# Interactive plot of bending moment and shear force
# Hover over the curves to see values at each node
fig_ss = plot_beam_diagrams(
    beam=ss_beam,
    title="Simply Supported Beam - M and V Diagrams<br>L=8m, w=25kN/m",
    height=600,
)
fig_ss.show()

## 4. Apply Tension Shift using the API <a id='tension-shift'></a>

Now we apply tension shift using the actual `MNInteractionDiagram.apply_tension_shift()` API.

### Case 1: Without Shear Reinforcement
- Shift distance: $a_l = d$ (effective depth)

The API:
1. Solves strain compatibility to find the lever arm z
2. Uses effective depth d for shift distance
3. Calculates $M_{add} = |V_{Ed}| \cdot a_l$
4. Caps the shifted moment at M_cap

In [9]:
# Apply tension shift WITHOUT shear reinforcement using the actual API
# This calls diagram.apply_tension_shift() for each node along the beam

ss_shifted_no_shear, ss_shift_dist, ss_cot_theta = apply_tension_shift_to_beam(
    beam=ss_beam,
    diagram=diagram,
    N_Ed=0.0,  # No axial force
    shear_reinforcement=None,  # No shear reinforcement → a_l = d
    iterate_z=False,
)

print("Tension Shift Results (WITHOUT shear reinforcement)")
print("=" * 55)
print(f"Shift distance a_l = d = {ss_shift_dist[0]:.1f} mm")
print(f"cot(θ) = {ss_cot_theta[0]} (None = no shear reinforcement)")
print()

# Compare at key locations
print("Effect at key locations:")
print(f"{'x (m)':<8} {'M_orig (kN·m)':<15} {'V (kN)':<12} {'M_design (kN·m)':<18} {'Increase':<10}")
print("-" * 70)

for x_check in [0.0, 1.0, 2.0, 4.0, 6.0, 8.0]:
    idx = np.argmin(np.abs(ss_beam.x_values - x_check))
    M_orig = ss_beam.M_values[idx]
    V = ss_beam.V_values[idx]
    M_design = ss_shifted_no_shear[idx]
    increase = (M_design - M_orig) if M_orig != 0 else 0
    print(f"{ss_beam.x_values[idx]:<8.2f} {M_orig:<15.2f} {V:<12.2f} {M_design:<18.2f} {increase:+.2f}")

Tension Shift Results (WITHOUT shear reinforcement)
Shift distance a_l = d = 440.0 mm
cot(θ) = None (None = no shear reinforcement)

Effect at key locations:
x (m)    M_orig (kN·m)   V (kN)       M_design (kN·m)    Increase  
----------------------------------------------------------------------
0.00     0.00            100.00       44.00              +0.00
0.96     84.48           76.00        117.92             +33.44
2.00     150.00          50.00        172.00             +22.00
4.00     200.00          0.00         200.00             +0.00
6.00     150.00          -50.00       172.00             +22.00
8.00     0.00            -100.00      44.00              +0.00


In [10]:
# Visualize: Original vs Shifted moments (no shear reinforcement)
fig_ss_shift = plot_tension_shift_comparison(
    beam=ss_beam,
    shifted_moments=ss_shifted_no_shear,
    shift_distances=ss_shift_dist,
    title="Simply Supported Beam - Tension Shift (a_l = d)<br>EC2 §9.2.1.3 without shear reinforcement",
    height=500,
)
fig_ss_shift.show()

### Case 2: With Shear Reinforcement

- Shift distance: $a_l = 0.5 \cdot z \cdot \cot\theta$
- The API calculates cot(θ) from shear design equations

In [11]:
# Create shear reinforcement (vertical stirrups)
shear_rebar = ShearRebar(
    grade="B500B",
    diameter=10,
    n_legs=2,
    spacing=200,
    angle=90,  # Vertical stirrups
)

print("Shear Reinforcement:")
print(f"  Configuration: {shear_rebar.n_legs}×ϕ{shear_rebar.diameter}@{shear_rebar.spacing}")
print(f"  A_sw/s = {shear_rebar.area_per_unit_length:.1f} mm²/m")
print(f"  Stirrup angle α = {shear_rebar.angle}° (vertical)")

Shear Reinforcement:
  Configuration: 2×ϕ10.0@200.0
  A_sw/s = 0.8 mm²/m
  Stirrup angle α = 90.0° (vertical)


In [12]:
# Apply tension shift WITH shear reinforcement using the actual API
ss_shifted_with_shear, ss_shift_dist_shear, ss_cot_theta_shear = apply_tension_shift_to_beam(
    beam=ss_beam,
    diagram=diagram,
    N_Ed=0.0,
    shear_reinforcement=shear_rebar,  # With shear reinforcement → a_l = 0.5·z·cot(θ)
    iterate_z=True,
)


print("Tension Shift Results (WITH shear reinforcement)")
print("=" * 55)
# Get typical values (non-None)
typical_idx = len(ss_beam.nodes) // 4  # Quarter span
cot_val = ss_cot_theta_shear[typical_idx]
theta_deg = np.degrees(np.arctan(1/cot_val)) if cot_val else None

print(f"cot(θ) = {cot_val:.3f}" if cot_val else "cot(θ) = N/A")
print(f"θ = {theta_deg:.1f}°" if theta_deg else "θ = N/A")
print(f"Shift distance a_l = 0.5·z·cot(θ) = {ss_shift_dist_shear[typical_idx]:.1f} mm")
print()

# Compare shift distances
print("Comparison of shift distances:")
print(f"  Without shear reinf.: a_l = d = {ss_shift_dist[typical_idx]:.0f} mm")
print(f"  With shear reinf.: a_l = 0.5·z·cot(θ) = {ss_shift_dist_shear[typical_idx]:.0f} mm")

Tension Shift Results (WITH shear reinforcement)
cot(θ) = 2.500
θ = 21.8°
Shift distance a_l = 0.5·z·cot(θ) = 495.0 mm

Comparison of shift distances:
  Without shear reinf.: a_l = d = 440 mm
  With shear reinf.: a_l = 0.5·z·cot(θ) = 495 mm


In [13]:
# Visualize: Original vs Shifted moments (with shear reinforcement)
cot_display = f"{cot_val:.2f}" if cot_val else "N/A"
fig_ss_shift2 = plot_tension_shift_comparison(
    beam=ss_beam,
    shifted_moments=ss_shifted_with_shear,
    shift_distances=ss_shift_dist_shear,
    title=f"Simply Supported Beam - Tension Shift (a_l = 0.5·z·cotθ)<br>EC2 §9.2.1.3 with shear reinforcement, cotθ≈{cot_display}",
    height=500,
)
fig_ss_shift2.show()

## 5. Comparison Visualization <a id='comparison'></a>

Let's compare both cases side by side.

In [14]:
import plotly.graph_objects as go

# Create comparison plot with all three moment envelopes
fig_compare = go.Figure()

# Original moment
fig_compare.add_trace(go.Scatter(
    x=ss_beam.x_values,
    y=-ss_beam.M_values,  # Negative for structural convention
    mode="lines",
    name="Original Moment",
    line=dict(color="blue", width=2),
    fill="tozeroy",
    fillcolor="rgba(0, 100, 255, 0.15)",
))

# Shifted without shear reinforcement
fig_compare.add_trace(go.Scatter(
    x=ss_beam.x_values,
    y=-ss_shifted_no_shear,
    mode="lines",
    name="Shifted (no shear reinf., a_l=d)",
    line=dict(color="orange", width=2, dash="dash"),
))

# Shifted with shear reinforcement
fig_compare.add_trace(go.Scatter(
    x=ss_beam.x_values,
    y=-ss_shifted_with_shear,
    mode="lines",
    name="Shifted (with shear reinf., a_l=0.5·z·cotθ)",
    line=dict(color="red", width=2, dash="dot"),
))

fig_compare.update_layout(
    title=dict(text="Comparison: Original vs Shifted Moments<br>Simply Supported Beam, L=8m, w=25kN/m", x=0.5),
    height=500,
    xaxis_title="Position along beam (m)",
    yaxis_title="Moment (kN·m) [↑ hogging, ↓ sagging]",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    hovermode="x unified",
)

fig_compare.add_hline(y=0, line_dash="solid", line_color="black", line_width=1)
fig_compare.show()

In [15]:
# Summary table
print("\nSummary: Simply Supported Beam Tension Shift")
print("=" * 70)
print(f"{'Parameter':<35} {'No Shear Reinf.':<18} {'With Shear Reinf.':<18}")
print("-" * 70)
print(f"{'Shift distance a_l (mm)':<35} {ss_shift_dist[typical_idx]:<18.0f} {ss_shift_dist_shear[typical_idx]:<18.0f}")
print(f"{'cot(θ)':<35} {'N/A':<18} {cot_val:<18.3f}" if cot_val else f"{'cot(θ)':<35} {'N/A':<18} {'N/A':<18}")
print(f"{'Max M_design (kN·m)':<35} {np.max(ss_shifted_no_shear):<18.2f} {np.max(ss_shifted_with_shear):<18.2f}")
print(f"{'M_design at x=2m (kN·m)':<35} {ss_shifted_no_shear[np.argmin(np.abs(ss_beam.x_values - 2))]:<18.2f} {ss_shifted_with_shear[np.argmin(np.abs(ss_beam.x_values - 2))]:<18.2f}")


Summary: Simply Supported Beam Tension Shift
Parameter                           No Shear Reinf.    With Shear Reinf. 
----------------------------------------------------------------------
Shift distance a_l (mm)             440                495               
cot(θ)                              N/A                2.500             
Max M_design (kN·m)                 200.00             200.00            
M_design at x=2m (kN·m)             172.00             174.75            


## 6. Fixed-Pinned Beam with Shear Reinforcement <a id='fixed-pinned'></a>

A fixed-pinned beam (propped cantilever) demonstrates tension shift for both:
- Positive (sagging) moments in the span
- Negative (hogging) moments at the fixed support

In [16]:
# Create fixed-pinned beam
fp_beam = FixedPinnedBeam(
    length_m=8.0,
    udl_kN_m=25.0,
    n_nodes=101,
)

print("Fixed-Pinned Beam Analysis")
print("=" * 40)
print(f"Span: {fp_beam.length_m} m")
print(f"UDL: {fp_beam.udl_kN_m} kN/m")
print()

M_max_pos, x_pos = fp_beam.get_max_positive_moment()
M_max_neg, x_neg = fp_beam.get_max_negative_moment()

print(f"Maximum positive moment: {M_max_pos:.2f} kN·m at x = {x_pos:.2f} m")
print(f"Maximum negative moment: {M_max_neg:.2f} kN·m at x = {x_neg:.2f} m")
print()
print(f"M_cap (positive): {fp_beam.get_M_cap_positive():.2f} kN·m")
print(f"M_cap (negative): {fp_beam.get_M_cap_negative():.2f} kN·m")

Fixed-Pinned Beam Analysis
Span: 8.0 m
UDL: 25.0 kN/m

Maximum positive moment: 112.48 kN·m at x = 4.96 m
Maximum negative moment: -200.00 kN·m at x = 0.00 m

M_cap (positive): 112.48 kN·m
M_cap (negative): 200.00 kN·m


In [17]:
# Plot M and V diagrams for fixed-pinned beam
fig_fp = plot_beam_diagrams(
    beam=fp_beam,
    title="Fixed-Pinned Beam - M and V Diagrams<br>L=8m, w=25kN/m",
    height=600,
)
fig_fp.show()

In [18]:
# Apply tension shift to fixed-pinned beam WITH shear reinforcement
fp_shifted, fp_shift_dist, fp_cot_theta = apply_tension_shift_to_beam(
    beam=fp_beam,
    diagram=diagram,
    N_Ed=0.0,
    shear_reinforcement=shear_rebar,
    iterate_z=False,
)

print("Fixed-Pinned Beam - Tension Shift Results")
print("=" * 55)
print()
print("Effect at key locations:")
print(f"{'x (m)':<8} {'M_orig (kN·m)':<15} {'V (kN)':<12} {'M_design (kN·m)':<18}")
print("-" * 60)

for x_check in [0.0, 1.0, 2.0, 3.0, 5.0, 8.0]:
    idx = np.argmin(np.abs(fp_beam.x_values - x_check))
    M_orig = fp_beam.M_values[idx]
    V = fp_beam.V_values[idx]
    M_design = fp_shifted[idx]
    print(f"{fp_beam.x_values[idx]:<8.2f} {M_orig:<+15.2f} {V:<+12.2f} {M_design:<+18.2f}")

Fixed-Pinned Beam - Tension Shift Results

Effect at key locations:
x (m)    M_orig (kN·m)   V (kN)       M_design (kN·m)   
------------------------------------------------------------
0.00     -200.00         +125.00      -200.00           
0.96     -91.52          +101.00      -141.51           
2.00     +0.00           +75.00       +37.12            
2.96     +60.48          +51.00       +85.73            
4.96     +112.48         +1.00        +112.48           
8.00     +0.00           -75.00       +37.12            


In [19]:
# Visualize fixed-pinned beam tension shift
fig_fp_shift = plot_tension_shift_comparison(
    beam=fp_beam,
    shifted_moments=fp_shifted,
    shift_distances=fp_shift_dist,
    title="Fixed-Pinned Beam - Tension Shift<br>EC2 §9.2.1.3 with shear reinforcement",
    height=500,
)
fig_fp_shift.show()

## Summary

This notebook demonstrated the EC2 §9.2.1.3 tension shift rule using the actual `MNInteractionDiagram.apply_tension_shift()` API:

### Key API Method

```python
result = diagram.apply_tension_shift(
    M_Ed=moment_at_section,      # Applied moment (kN·m)
    V_Ed=shear_at_section,       # Applied shear (kN)
    N_Ed=axial_force,            # Applied axial force (kN)
    M_cap=capacity_moment,       # Maximum moment cap (kN·m)
    shear_reinforcement=...,     # ShearRebar or None
    iterate_z=False,             # True for iterative z solution
)
```

### Returns `TensionShiftResult` with:
- `M_design`: Design moment after shift and capping (kN·m)
- `M_add`: Additional moment from shift (kN·m)
- `shift_distance_a_l_mm`: Shift distance (mm)
- `cot_theta`: Strut angle cotangent (or None if no shear reinf.)

### Key Observations

1. **At supports**: High shear → large M_add → significant moment increase
2. **At midspan**: Low shear → small M_add → minimal effect
3. **With shear reinforcement**: Flatter strut angle (larger cot(θ)) → larger shift
4. **M_cap limits**: Shifted moment is capped at maximum moment in span

### References

- EN 1992-1-1:2004 §9.2.1.3 - Curtailment of longitudinal tension reinforcement
- EC2 Figure 9.2 - Illustration of tension shift rule